## Re-ranking por relevancia marginal máxima (MMR)

MMR es un método de re-ranking que combina la relevancia y la diversidad de los documentos recuperados. El objetivo es maximizar la relevancia de los documentos devueltos y minimizar la redundancia entre ellos.

Esto puede ser útil para incrementar la habilidad de los modelos de lenguage para generar respuestas con mayor cobertura y profundidad.

Su algoritmo es el siguiente:

1. Calcular los `embeddings` para cada documento y para la consulta.
2. Seleccionar el documento más relevante para la consulta.
3. Para cada documento restante, calcular el promedio de similitud de los documentos ya seleccionados.
4. Seleccionar el documento que es, en promedio, menos similar a los documentos ya seleccionados.
5. Repitir los pasos 3 y 4 hasta que se hayan seleccionado `k` documentos. Es decir, una lista ordenada que parte del documento que más contribuye a la diversidad general hasta el documento que contribuye menos.

En Langchain, el algoritmo de MMR es utilizado después de que el `retriever` ha recuperado los documentos más relevantes para la consulta. Por lo tanto, nos aseguramos que estamos seleccionando documentos diversos de un conjunto de documentos que ya son relevantes para la consulta.

![diagrama](../docs/media/diagrams/04.png)

## Librerías

In [1]:
from dotenv import load_dotenv
from langchain_mistralai import MistralAIEmbeddings
from langchain_chroma import Chroma

from src.langchain_docs_loader import load_langchain_docs_splitted

load_dotenv()

True

## Carga de datos

In [2]:
docs = load_langchain_docs_splitted()
len(docs)

557

## Creación de retriever

Normalmente creamos nuestro retriever de la siguiente manera:

In [3]:
similarity_retriever = Chroma.from_documents(
    documents=docs,
    embedding=MistralAIEmbeddings(),
).as_retriever(k=4)

Sin embargo, podrás notar que de hacerlo, el tipo de búsqueda que se realiza es por similitud de vectores. En este caso, queremos realizar una búsqueda por similitud de vectores, pero con un re-ranking por relevancia marginal máxima (MMR).

In [4]:
similarity_retriever.search_type

'similarity'

Entonces, para crear un retriever con re-ranking por MMR, debemos hacer lo siguiente:

In [5]:
mmr_retriever = Chroma.from_documents(
    documents=docs,
    embedding=MistralAIEmbeddings(),
).as_retriever(
    search_type="mmr",
    k=4,  # number of documents to retrieve after mmr
    fetch_k=20,  # number of documents to fetch in the first step
    # Lambda mult is a number between 0 and 1 that determines the degree
    # of diversity among the results with 0 corresponding to maximum diversity
    # and 1 to minimum diversity.
    lambda_mult=0.5,
)

Ahora nuestro retriever está listo para ser utilizado con re-ranking por MMR.

In [6]:
mmr_retriever.search_type

'mmr'

## Uso del retriever

In [8]:
similarity_retriever.invoke(
    "How to integrate LCEL into my Retrieval augmented generation system with a keyword search retriever?"
)

[Document(id='e637d32e-b227-4310-b868-dbb693794bc3', metadata={'title': 'Retrieval - Docs by LangChain', 'source': 'https://docs.langchain.com/oss/python/langchain/retrieval', 'language': 'en'}, page_content='Large Language Models (LLMs) are powerful, but they have two key limitations:\n\n- **Finite context** — they can’t ingest entire corpora at once.\n- **Static knowledge** — their training data is frozen at a point in time.\n\nRetrieval addresses these problems by fetching relevant external knowledge at query time. This is the foundation of **Retrieval-Augmented Generation (RAG)**: enhancing an LLM’s answers with context-specific information.\n\n## \u200bBuilding a knowledge base\n\nA **knowledge base** is a repository of documents or structured data used during retrieval.\n\nIf you need a custom knowledge base, you can use LangChain’s document loaders and vector stores to build one from your own data.\n\nIf you already have a knowledge base (e.g., a SQL database, CRM, or internal d

In [9]:
mmr_retriever.invoke(
    "How to integrate LCEL into my Retrieval augmented generation system with a keyword search retriever?"
)

[Document(id='e637d32e-b227-4310-b868-dbb693794bc3', metadata={'title': 'Retrieval - Docs by LangChain', 'language': 'en', 'source': 'https://docs.langchain.com/oss/python/langchain/retrieval'}, page_content='Large Language Models (LLMs) are powerful, but they have two key limitations:\n\n- **Finite context** — they can’t ingest entire corpora at once.\n- **Static knowledge** — their training data is frozen at a point in time.\n\nRetrieval addresses these problems by fetching relevant external knowledge at query time. This is the foundation of **Retrieval-Augmented Generation (RAG)**: enhancing an LLM’s answers with context-specific information.\n\n## \u200bBuilding a knowledge base\n\nA **knowledge base** is a repository of documents or structured data used during retrieval.\n\nIf you need a custom knowledge base, you can use LangChain’s document loaders and vector stores to build one from your own data.\n\nIf you already have a knowledge base (e.g., a SQL database, CRM, or internal d